# 🔍 Silver EDA — Exploratory Data Analysis
**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Source:** main.default.silver_warehouse_sales

### Business Questions
1. Which item types generate the most sales volume?
2. Is there seasonality — months or years with higher sales?
3. Who are the top suppliers by total sales?
4. Which products sell the most on average?
5. Do customers prefer retail or warehouse channel?
6. Which suppliers carry the most product variety?

In [0]:
import sys
from datetime import datetime
from pyspark.sql import functions as F

# Define table name once — change here if catalog/schema changes
SILVER_TABLE = "main.default.silver_warehouse_sales"

# Expected schema validation — fails loudly if Silver table changed
EXPECTED_COLS = [
    'date', 'year', 'month', 'supplier', 'item_code',
    'item_description', 'item_type', 'retail_sales',
    'retail_transfers', 'warehouse_sales', 'total_sales'
]

try:
    df = spark.table(SILVER_TABLE)

    # Schema validation
    missing = set(EXPECTED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in Silver table: {missing}")
    
    # Validación extra de columnas no esperadas
    extra = set(df.columns) - set(EXPECTED_COLS)
    if extra:
        print(f"  ⚠ Extra columns found: {extra}")

    # Get row count for summary
    row_count = df.count()

    print("=" * 50)
    print("  ENVIRONMENT & DATA SUMMARY")
    print("=" * 50)
    print(f"  Spark version  : {spark.version}")
    print(f"  Python version : {sys.version.split()[0]}")
    print(f"  Run timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Table          : {SILVER_TABLE}")
    print(f"  Rows           : {row_count:,}")
    print(f"  Columns        : {len(df.columns)}")
    print(f"  Schema         : validated ✓")
    print(f"  Compute        : Serverless (auto-optimized)")
    print("=" * 50)

except ValueError as ve:
    print(f"  ✗ Schema validation failed: {ve}")
    raise
except Exception as e:
    print(f"  ✗ Failed to load Silver table: {e}")
    raise

In [0]:
# Create cached clean DataFrame to avoid repeating filters
# Excludes REF, DUNNAGE and UNCLASSIFIED across all analyses
df_clean = df.filter(
    ~F.col("item_type").isin("REF", "DUNNAGE", "UNCLASSIFIED")
)

clean_count = df_clean.count()
print(f"Clean rows (excluding REF/DUNNAGE/UNCLASSIFIED): {clean_count:,}")

## Analysis 1 — Sales Distribution by Item Type
**Question:** Which item types generate the most sales volume?  
**Why it matters:** Understanding category dominance helps prioritize 
inventory and marketing decisions.

In [0]:
# Question 1: Which item types generate the most total sales?
# Using df_clean (already excludes REF, DUNNAGE and UNCLASSIFIED)

sales_by_type = df_clean \
    .groupBy("item_type") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales_per_transaction"),
        F.count("*").alias("transaction_count")
    ) \
    .orderBy(F.col("total_sales").desc())

display(sales_by_type)

### Key Findings — Item Type Distribution

> **Filtering note:** REF, DUNNAGE, and UNCLASSIFIED are excluded from 
> all analyses. REF and DUNNAGE represent returns and packaging adjustments 
> (negative values — not data errors). UNCLASSIFIED is the single null row 
> filled during Silver transformation. None of these represent real sales.

| Item Type | Total Sales | Avg per Transaction | Transactions |
|-----------|-------------|---------------------|--------------|
| BEER | 7,668,171 | 180.8 | 42,413 |
| WINE | 2,638,101 | 14.06 | 187,640 |
| LIQUOR | 1,692,333 | 26.07 | 64,910 |
| KEGS | 118,430 | 11.67 | 10,146 |
| NON-ALCOHOL | 86,900 | 45.55 | 1,908 |
| STR_SUPPLIES | 13,587 | 33.55 | 405 |

> **Unit clarification:** According to the official dataset description,
> values represent **sales and movement data** (units moved), not dollars.

> **Insight:** BEER dominates by volume per transaction (avg 180.8 units)
> while WINE dominates by transaction count (187,640 transactions).
> These are fundamentally different sales patterns — BEER moves in bulk,
> WINE moves in high frequency but smaller quantities.

## Analysis 2 — Sales Trends Over Time
**Question:** How have total sales evolved by year and month?  
**Why it matters:** Identifying seasonality and growth trends is essential
for forecasting and ML feature engineering.

In [0]:
# Question 2: How do sales trend over time?
# We group by year and month to see monthly totals
# This reveals seasonality patterns and year-over-year growth

monthly_sales = df_clean.groupBy("year", "month") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.sum("retail_sales"), 2).alias("retail_sales"),
        F.round(F.sum("warehouse_sales"), 2).alias("warehouse_sales"),
        F.count("*").alias("transaction_count")
    ) \
    .orderBy("year", "month")

display(monthly_sales)

Databricks visualization. Run in Databricks to view.

### Key Findings — Monthly Sales Trends

| Year | Months Available | Coverage |
|------|-----------------|---------|
| 2017 | Jun–Dec | 7 of 12 months |
| 2018 | Jan–Feb | 2 of 12 months |
| 2019 | Jan–Nov | 11 of 12 months |
| 2020 | Jan, Mar, Jul, Sep | 4 of 12 months |

- **2019 is the most reliable year** — 11 consecutive months of data
- **2018 is nearly unusable** — only 2 months available
- **2020 gaps coincide with COVID-19 pandemic period** — cause unconfirmed, further investigation needed
- **Seasonality detected:** December (2017) and July (2020) show peak sales
- **Warehouse sales always exceeds retail** — primary channel is B2B

> ⚠️ **ML implication:** Dataset is not continuous. Missing months are 
> absent from the source — not zero sales. Year cannot be used directly 
> as a feature without careful handling. Consider focusing ML modeling 
> on 2019 data only for the most reliable results.

## Analysis 3 — Top Suppliers by Total Sales
**Question:** Which suppliers generate the most sales volume?  
**Why it matters:** Identifying key suppliers helps understand 
market concentration and dependency risks.

In [0]:
# Question 3: Which suppliers generate the most total sales?
# We exclude UNKNOWN suppliers (rows where supplier was null in Bronze)
# top 15 gives enough coverage without overwhelming the visualization

top_suppliers = df_clean.filter(F.col("supplier") != "UNKNOWN") \
    .groupBy("supplier") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales_per_transaction"),
        F.count("*").alias("transaction_count"),
        F.countDistinct("item_type").alias("item_types_carried")
    ) \
    .orderBy(F.col("total_sales").desc()) \
    .limit(15)

display(top_suppliers)

Databricks visualization. Run in Databricks to view.

In [0]:
# Calculate exact market concentration of top 3 suppliers
# This quantifies the supplier dependency risk mentioned in the markdown

total_market_row = df_clean.agg(
    F.sum("total_sales").alias("total_market_sales")
).collect()[0]

total_market = total_market_row["total_market_sales"] or 0  # ✅ Protección None

top3_sales_row = df.filter(
    F.col("supplier").isin(
        "CROWN IMPORTS",
        "MILLER BREWING COMPANY",
        "ANHEUSER BUSCH INC"
    )
).agg(F.sum("total_sales").alias("top3_sales")).collect()[0]

top3_sales = top3_sales_row["top3_sales"] or 0  # ✅ Protección None

# ✅ Protección explícita contra división por cero
concentration_pct = round(top3_sales / total_market * 100, 1) if total_market > 0 else 0

print("=" * 50)
print("  SUPPLIER CONCENTRATION ANALYSIS")
print("=" * 50)
print(f"  Total market volume : {total_market:,.0f} units")
print(f"  Top 3 suppliers     : {top3_sales:,.0f} units")
print(f"  Market concentration: {concentration_pct}%")
print("=" * 50)

### Key Findings — Top Suppliers

| Supplier | Total Sales | Category |
|----------|-------------|---------|
| Crown Imports | \~2.0M | BEER (Corona, Modelo) |
| Miller Brewing Company | \~1.6M | BEER (Miller, Coors) |
| Anheuser Busch Inc | \~1.5M | BEER (Budweiser, Bud Light) |
| Heineken USA | \~1.1M | BEER |
| E & J Gallo Winery | \~1.0M | WINE |

> **Insight:** The top 3 suppliers are all beer companies and account for 
> the majority of total volume. This confirms that BEER's dominance 
> in volume is driven by a small number of large distributors moving 
> product in bulk — not by high transaction frequency.

> **Market concentration:** The top 3 suppliers (Crown Imports, Miller Brewing,
> Anheuser Busch) control **40.6% of total market volume** (4,966,314 of 
> 12,217,524 units). Any supply chain disruption from these three companies 
> would significantly impact Montgomery County's total sales.

## Analysis 4 — Retail vs Warehouse Sales Channel
**Question:** Do customers prefer retail or warehouse channel?  
**Why it matters:** Understanding channel distribution helps Montgomery 
County optimize inventory allocation between stores and warehouse.

In [0]:
# Question 4: How do retail and warehouse channels compare?
# Using df_clean (already excludes REF, DUNNAGE and UNCLASSIFIED)
# Division by zero protection added — if total_sales = 0, percentage = 0

channel_comparison = df_clean \
    .groupBy("item_type") \
    .agg(
        F.round(F.sum("retail_sales"), 2).alias("total_retail"),
        F.round(F.sum("retail_transfers"), 2).alias("total_transfers"),
        F.round(F.sum("warehouse_sales"), 2).alias("total_warehouse"),
        F.round(F.sum("total_sales"), 2).alias("total_sales")
    ) \
    .withColumn("retail_pct",
        F.when(F.col("total_sales") == 0, 0)
        .otherwise(F.round(F.col("total_retail") / F.col("total_sales") * 100, 1))
    ) \
    .withColumn("transfers_pct",
        F.when(F.col("total_sales") == 0, 0)
        .otherwise(F.round(F.col("total_transfers") / F.col("total_sales") * 100, 1))
    ) \
    .withColumn("warehouse_pct",
        F.when(F.col("total_sales") == 0, 0)
        .otherwise(F.round(F.col("total_warehouse") / F.col("total_sales") * 100, 1))
    ) \
    .orderBy(F.col("total_sales").desc())

display(channel_comparison)

### Key Findings — Sales Channel Analysis

| Item Type | Retail % | Transfers % | Warehouse % |
|-----------|---------|------------|-------------|
| BEER | 7.5% | 7.4% | 85.1% |
| WINE | 28.3% | 27.8% | 43.9% |
| LIQUOR | 47.4% | 47.0% | 5.6% |
| KEGS | 0% | 0% | 100% |
| NON-ALCOHOL | 39.2% | 30.7% | 30.1% |

> **Retail Transfers** represent stock moved between county stores — 
> not new sales. LIQUOR has a surprisingly high transfer rate (47%), 
> suggesting frequent rebalancing of inventory between retail locations.

> **Key insight:** BEER is bulk B2B. LIQUOR is primarily retail consumer
> with high inter-store transfers. WINE is balanced across all three channels.

> **ML implication:** Each item_type has a completely different channel 
> pattern. Use item_type as a strong categorical feature with one-hot encoding.

## Analysis 5 — Top Products by Sales Volume
**Question:** Which individual products sell the most?  
**Why it matters:** Category-level analysis tells us what type sells most,
but product-level analysis tells us which specific items drive that volume.
This is actionable for inventory and procurement decisions.

In [0]:
# Question 5: Which individual products sell the most?
# Using df_clean (already excludes REF, DUNNAGE and UNCLASSIFIED)
# item_description + item_type + supplier gives full product context

top_products = df_clean \
    .groupBy("item_description", "item_type", "supplier") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.sum("retail_sales"), 2).alias("total_retail"),
        F.round(F.sum("warehouse_sales"), 2).alias("total_warehouse"),
        F.count("*").alias("transaction_count")
    ) \
    .orderBy(F.col("total_sales").desc()) \
    .limit(20)

display(top_products)

### Key Findings — Top 20 Products

**All top 20 products are BEER.** No wine, liquor, or any other category 
appears in the top 20 by total volume.

| Rank | Product | Supplier | Total Sales | Channel |
|------|---------|----------|-------------|---------|
| 1 | Corona Extra Loose NR - 12oz | Crown Imports | 352,574 | 86% warehouse |
| 2 | Corona Extra 2/12 NR - 12oz | Crown Imports | 266,992 | 93% warehouse |
| 3 | Heineken Loose NR - 12oz | Heineken USA | 206,675 | 83% warehouse |
| 4 | Heineken 2/12 NR - 12oz | Heineken USA | 169,564 | 91% warehouse |
| 5 | Miller Lite 30pk Can - 12oz | Miller Brewing | 162,971 | 82% warehouse |

> **Insight:** Corona Extra alone accounts for 352,574 units — 
> nearly 5% of all BEER volume. The top 5 products are all 
> imported or premium beers sold almost exclusively through 
> the warehouse channel to bars and restaurants.

> **Supplier concentration:** Crown Imports (Corona, Modelo) appears 
> 7 times in the top 20. Miller Brewing appears 5 times. 
> Anheuser Busch appears 3 times. Heineken appears 3 times.
> These 4 suppliers own the top 20 entirely.

> **ML implication:** Product identity (item_description) is a strong 
> predictor — top products have consistent transaction counts of exactly 
> 24 months, confirming they were sold every single month of the dataset.